# freqk_gr — ΔAF analysis (field vs seed-mix founder)

Pipeline:
1. Load seed-mix k-mer counts → founder baseline AF per variant  
2. Load field samples (lean) → compute Δp = field AF − baseline  
3. Pivot to **wide matrix** (variants × samples) of Δp  
4. Sample a subset for quick EDA without crashing the notebook  
5. Analysis: QC, temporal trajectories, per-plot comparisons

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

RESULTS = Path("/home/tbellagio/scratch/freqk_gr/results")
DATA    = Path("/home/tbellagio/scratch/freqk_gr/data")
K = 31

# How many variants to sample for quick EDA plots (set None to use all)
N_SAMPLE = 50_000


## 1. Discover files

In [ ]:
af_files = sorted(RESULTS.glob(f"*/*/k{K}/*.counts_by_allele.k{K}.tsv"))
sm_files = sorted(RESULTS.glob(f"seedmix/*/k{K}/*.counts_by_allele.k{K}.tsv"))

# Exclude seed-mix from field files (they live under results/seedmix/)
af_files = [f for f in af_files if "seedmix" not in f.parts]

print(f"Field samples : {len(af_files)}")
for f in af_files:
    print(" ", f.parent.parent.name)
print(f"\nSeed mix reps : {len(sm_files)}")
for f in sm_files:
    print(" ", f.parent.parent.name)


## 2. Seed-mix baseline

Average AF across S1–S8 per variant.  
Infinite-population assumption for the founder pool — just take the mean.  
Require coverage in ≥ `MIN_SM_REPS` of the 8 replicates.

In [ ]:
MIN_SM_REPS = 4   # out of 8

def load_counts(path):
    """Minimal loader: returns DataFrame with ref|alt counts + variant_idx."""
    df = pd.read_csv(path, sep="|", header=None, names=["ref","alt"], dtype=float)
    df[["ref","alt"]] = df[["ref","alt"]].fillna(0).astype(int)
    df["variant_idx"] = df.index
    total = df["ref"] + df["alt"]
    df["af"] = np.where(total > 0, df["alt"] / total, np.nan)
    df["coverage_kmers"] = total
    return df

if not sm_files:
    raise RuntimeError("No seed-mix results yet — run: python scripts/launch_seedmix.py")

sm_dfs = []
for f in sm_files:
    tmp = load_counts(f)
    tmp["replicate"] = f.parent.parent.name   # SEEDMIX_S1 …
    sm_dfs.append(tmp[tmp["coverage_kmers"] > 0][["variant_idx","replicate","af"]])

sm_all = pd.concat(sm_dfs, ignore_index=True)

sm_baseline = (
    sm_all.groupby("variant_idx")["af"]
    .agg(af_sm="mean", af_sm_std="std", n_sm_reps="count")
    .reset_index()
)
sm_baseline = sm_baseline[sm_baseline["n_sm_reps"] >= MIN_SM_REPS].copy()

print(f"Seed-mix baseline variants (≥{MIN_SM_REPS}/8 reps covered): {len(sm_baseline):,}")
print(f"Mean cross-replicate std: {sm_baseline['af_sm_std'].mean():.4f}")
sm_baseline.head()


## 3. Load field samples (lean)

Load only the columns we need; filter immediately to variants that have a  
seed-mix baseline *and* are covered in this sample.  
This keeps the tall DataFrame small.

In [ ]:
def parse_sample_id(sample_id):
    site = sample_id[4:6]
    plot = sample_id[6:8]
    date = sample_id[8:16]
    year = sample_id[8:12]
    return site, plot, date, year

sm_idx = set(sm_baseline["variant_idx"])  # fast lookup set

rows = []
for path in af_files:
    sid = path.parent.parent.name
    site, plot, date, year = parse_sample_id(sid)
    df = load_counts(path)
    # keep only variants with seed-mix baseline AND k-mer coverage
    df = df[(df["coverage_kmers"] > 0) & (df["variant_idx"].isin(sm_idx))].copy()
    df["sample_id"] = sid
    df["site"]      = site
    df["plot"]      = plot
    df["date"]      = pd.to_datetime(date, format="%Y%m%d")
    df["year"]      = year
    rows.append(df[["variant_idx","sample_id","site","plot","date","year","af","coverage_kmers"]])
    print(f"  {sid}: {len(df):,} covered variants with baseline")

df_tall = pd.concat(rows, ignore_index=True)
print(f"\nTall DataFrame: {df_tall.shape}  ({df_tall['variant_idx'].nunique():,} unique variants, {df_tall['sample_id'].nunique()} samples)")


## 4. Join sample metadata

In [ ]:
meta = pd.read_csv(DATA / "samples_data_fix57.csv", usecols=[
    "sampleid","flowerscollected","generation","fix_57_generation",
    "coverage","weighted_mean_coverage","usesample"
]).rename(columns={"sampleid":"sample_id"})

df_tall = df_tall.merge(meta, on="sample_id", how="left")
df_tall["N"]      = df_tall["flowerscollected"]
df_tall["min_af"] = 1 / df_tall["N"]

print(df_tall.dtypes)
df_tall.head()


## 5. Compute Δp = field AF − seed-mix baseline

Join baseline per `variant_idx`, then compute the allele-frequency shift.

In [ ]:
df_tall = df_tall.merge(
    sm_baseline[["variant_idx","af_sm","af_sm_std","n_sm_reps"]],
    on="variant_idx", how="left"
)
df_tall["delta_p"]     = df_tall["af"] - df_tall["af_sm"]
df_tall["abs_delta_p"] = df_tall["delta_p"].abs()

print(f"delta_p range: {df_tall['delta_p'].min():.3f} to {df_tall['delta_p'].max():.3f}")
print(f"Mean |Δp|: {df_tall['abs_delta_p'].mean():.4f}")
df_tall[["variant_idx","sample_id","af","af_sm","delta_p"]].head()


## 6. Wide Δp matrix  (variants × samples)

Rows = `variant_idx`, columns = `sample_id`, values = `delta_p`.  
This is the primary object for downstream analysis.

In [ ]:
dp_wide = df_tall.pivot_table(
    index="variant_idx", columns="sample_id", values="delta_p"
)
dp_wide.columns.name = None

print(f"Wide matrix shape: {dp_wide.shape}")
print(f"Memory: {dp_wide.memory_usage(deep=True).sum() / 1e6:.1f} MB")
dp_wide.head()


## 7. Sample for quick EDA

Draw `N_SAMPLE` random variants so plots render fast.  
Set `N_SAMPLE = None` at the top to use the full matrix.

In [ ]:
rng = np.random.default_rng(42)

if N_SAMPLE and N_SAMPLE < len(dp_wide):
    sample_idx = rng.choice(dp_wide.index, size=N_SAMPLE, replace=False)
    dp_sample  = dp_wide.loc[sample_idx]
    tall_sample = df_tall[df_tall["variant_idx"].isin(sample_idx)]
    print(f"Sampled {N_SAMPLE:,} / {len(dp_wide):,} variants for EDA")
else:
    dp_sample   = dp_wide
    tall_sample = df_tall
    print("Using full matrix for EDA")


## 8. QC

In [ ]:
# Coverage and below-min-AF summary per sample
qc = (
    df_tall.groupby("sample_id")
    .apply(lambda x: pd.Series({
        "n_variants":   len(x),
        "mean_cov":     x["coverage_kmers"].mean(),
        "pct_low_cov":  100 * (x["coverage_kmers"] < 5).mean(),
        "mean_abs_dp":  x["abs_delta_p"].mean(),
    }))
    .reset_index()
)
print(qc.to_string(index=False))


In [ ]:
# Δp distribution per sample (sampled variants)
fig, ax = plt.subplots(figsize=(10, 4))
for sid, grp in tall_sample.groupby("sample_id"):
    grp["delta_p"].dropna().plot.hist(bins=60, alpha=0.35, density=True, label=sid, ax=ax)
ax.axvline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Δp  (field − seed mix)")
ax.set_ylabel("Density")
ax.set_title("Δp distribution per sample")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()


## 9. Δp over time — site 04

In [ ]:
site04 = tall_sample[tall_sample["site"] == "04"].copy()

# mean Δp per date (averaged across variants and plots)
summary = (
    site04.groupby(["date","generation"])["delta_p"]
    .agg(mean="mean", std="std")
    .reset_index()
)

gen_colors = {1:"steelblue", 2:"darkorange", 3:"forestgreen"}

fig, ax = plt.subplots(figsize=(8, 4))
for gen, grp in summary.groupby("generation"):
    ax.errorbar(grp["date"], grp["mean"], yerr=grp["std"],
                fmt="o-", capsize=4, color=gen_colors.get(gen,"gray"),
                label=f"Gen {int(gen)}")
ax.axhline(0, color="black", lw=1, ls="--", label="seed mix baseline")
ax.set_xlabel("Date")
ax.set_ylabel("Mean Δp")
ax.set_title("Site 04 — mean Δp over time")
ax.legend(title="Generation")
ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%Y-%m-%d"))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


## 10. Per-plot Δp comparison — site 04

In [ ]:
site04["date_gen"] = site04.apply(
    lambda r: (
        f"{r['date'].strftime('%Y-%m-%d')}\n(Gen {int(r['generation'])})"
        if pd.notna(r["generation"]) else r["date"].strftime("%Y-%m-%d")
    ), axis=1
)

fig, ax = plt.subplots(figsize=(11, 4))
sns.boxplot(
    data=site04, x="date_gen", y="delta_p", hue="plot",
    ax=ax, flierprops=dict(marker=".", alpha=0.2, markersize=1)
)
ax.axhline(0, color="black", lw=1, ls="--")
ax.set_xlabel("Date (Generation)")
ax.set_ylabel("Δp  (field − seed mix)")
ax.set_title("Site 04 — Δp per plot per time point")
ax.legend(title="Plot", bbox_to_anchor=(1.01,1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()


## 11. Filter to variants that changed

Keep only variants where `|Δp| > threshold` in ≥ 1 sample.  
Threshold = `max(1/N, DELTA_MIN)` — respects the per-sample flower-count noise floor.

In [ ]:
DELTA_MIN = 0.05

df_tall["threshold"] = np.maximum(df_tall["min_af"].fillna(DELTA_MIN), DELTA_MIN)
df_tall["is_changed"] = df_tall["abs_delta_p"] > df_tall["threshold"]

changed_idx = (
    df_tall.groupby("variant_idx")["is_changed"].any()
)
changed_idx = changed_idx[changed_idx].index

print(f"Total tested variants : {df_tall['variant_idx'].nunique():,}")
print(f"Changed in ≥1 sample  : {len(changed_idx):,}  ({100*len(changed_idx)/df_tall['variant_idx'].nunique():.1f}%)")

dp_changed = dp_wide.loc[dp_wide.index.isin(changed_idx)]
print(f"Changed wide matrix   : {dp_changed.shape}")


In [ ]:
# Save for downstream use
out_path = RESULTS / "site04_delta_p.tsv"
# melt changed wide matrix back to tall for saving
dp_changed.reset_index().melt(id_vars="variant_idx", var_name="sample_id", value_name="delta_p")    .dropna(subset=["delta_p"])    .to_csv(out_path, sep="\t", index=False)
print(f"Saved → {out_path}")
